# BayesNF — ablation геофич (Spatial-input layer)

**Цель.** При фиксированной сезонности (две лучшие из time_seasonality-ablation:
**WY_h1_10** и **Y_h10**) понять, какие пространственные/геофизические фичи
действительно полезны, а какие — шум.

**Что варьируем (5 вариантов).**

| Вариант      | Доп. фичи                          | Гипотеза                                |
|--------------|------------------------------------|------------------------------------------|
| `base`       | — (lat/lon/elev + 6 soil)          | Текущий baseline (после likelihood-абл.) |
| `proj`       | `x_proj, y_proj`                   | EPSG:3035 даёт прямые расстояния         |
| `idw`        | `idw`                              | Дешёвый FFRK — соседи как 1 скаляр       |
| `svd5`       | `svd_05..svd_09`                   | Сжатое поле корреляций (KL-truncated)    |
| `ffrk_full`  | `idw, gos, svd_05..svd_09`         | Полный FFRK-блок                         |

**Что фиксируем.**
- Likelihood: ZINB, target `rainfall_int = round(10·mm)`
- Архитектура: width=256, depth=2, 100 epochs, ensemble=4, lr=0.01
- Период: 2018-2022, fold 0
- Сезонность: **WY_h1_10** (победитель) и **Y_h10** (контроль) — две лучшие
  конфигурации из time_seasonality-ablation

**Зачем сравнивать с двумя сезонностями.** Если эффект геофич сохраняется и
для Y_h10, и для WY_h1_10 — значит, он "ортогонален" сезонности и реален. Если
эффект виден только у одной — есть взаимодействие, и FFRK "лечит" то же, что и
weekly harmonic (high-freq variability).

**Метрики.** CRPS (3 квантиля 5/50/95), MAE, RMSE, bias на всех точках и на wet-only
($y \ge 0.5$ mm).

**Структура.** Setup + helper `train_eval(...)` — выполняешь один раз сверху. Дальше
ПО ОДНОЙ ЯЧЕЙКЕ на эксперимент. Секция 7 — WY_h1_10-track, секция 8 — Y_h10-track.


## 1. Окружение + импорты


In [ ]:
import os
# JAX env vars ДОЛЖНЫ быть установлены ДО первого `import jax`
os.environ.setdefault('JAX_LOG_COMPILES', '0')
os.environ.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION', '0.90')

import sys
import gc
import time
import json
import threading
from pathlib import Path

import numpy as np
import pandas as pd

import jax
import jax.numpy as jnp
import bayesnf
from bayesnf.spatiotemporal import BayesianNeuralFieldVI

jax.config.update('jax_default_matmul_precision', 'tensorfloat32')

# Repo root: notebooks/05_bayesnf/geo_features/<этот>.ipynb → ../../..
ROOT = Path('../../..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

print(f'cwd      : {os.getcwd()}')
print(f'bayesnf  : {getattr(bayesnf, "__version__", "n/a")}')


## 2. GPU-диагностика


In [ ]:
import subprocess

backend = jax.default_backend()
devices = jax.devices()
print(f'JAX backend : {backend}')
print(f'JAX devices : {devices}')
print(f'n local     : {jax.local_device_count()}')

if backend != 'gpu':
    print('\n⚠ JAX на CPU — этот ноутбук рассчитан на GPU.')

try:
    smi = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.used,memory.total',
         '--format=csv,noheader'],
        capture_output=True, text=True, timeout=5,
    )
    print('\nnvidia-smi:')
    print(smi.stdout)
except Exception as e:
    print(f'nvidia-smi unavailable: {e}')


## 3. Конфиг эксперимента

Фиксируем likelihood (**ZINB**), архитектуру, годы. Сезонность и геофичи
передаются в `train_eval(...)` per-cell.

**Доступные колонки в parquet** (для справки):
- Spatial: `latitude, longitude, x_proj, y_proj, elevation_m`
- FFRK: `idw, gos, svd_00..svd_20`
- Soil: `bulk_density, clay, sand, silt, soc, water_10kpa`

**Две лучшие сезонности** (из time_seasonality-ablation):
- `WY_h1_10`: `seasonality_periods=['W','Y']`, `num_seasonal_harmonics=[1,10]`
- `Y_h10`:    `seasonality_periods=['Y']`,     `num_seasonal_harmonics=[10]`


In [ ]:
# --- Likelihood и target ---
LIKELIHOOD    = 'ZINB'
TARGET_COL    = 'rainfall_int'
NEEDS_RESCALE = True
PRECIP_SCALE  = 10

# --- Период данных (5 лет) ---
FOLD       = 0
YEAR_START = 2018
YEAR_END   = 2022

# --- Две лучшие сезонности из time_seasonality-ablation ---
SEAS_WY_H1_10 = {'periods': ['W', 'Y'], 'harmonics': [1, 10]}  # победитель
SEAS_Y_H10    = {'periods': ['Y'],      'harmonics': [10]}     # control

# --- Архитектура (фикс для всех ablation-вариантов) ---
WIDTH         = 256
DEPTH         = 2
NUM_EPOCHS    = 100
BATCH_SIZE    = 131072
LEARNING_RATE = 0.01
KL_WEIGHT     = 0.1
ENSEMBLE_SIZE = 4
SEED          = 0
SAMPLE_SIZE_POSTERIOR = 5   # default bayesnf=30 → 5 даёт ~6× speedup на predict

# --- Output ---
OUT_DIR = Path('results/bayesnf/experiments/geo_features')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'likelihood   : {LIKELIHOOD}  (target={TARGET_COL}, rescale={NEEDS_RESCALE})')
print(f'years        : {YEAR_START}-{YEAR_END}, fold {FOLD}')
print(f'seasonalities: WY_h1_10 = {SEAS_WY_H1_10}')
print(f'               Y_h10    = {SEAS_Y_H10}')
print(f'out dir      : {OUT_DIR}')


## 4. Загрузка данных (5 лет, fold 0)

Грузим ВСЕ потенциально-нужные колонки сразу. На стадии train_eval каждый
эксперимент возьмёт свой subset через `feature_cols`.


In [ ]:
S3_BUCKET   = 'thesis-data-ismaktam'
DATA_S3_DIR = 'bayesnf/data'
DATA_DIR    = Path('results/bayesnf/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_path = DATA_DIR / f'bayesnf_fold{FOLD}_train.parquet'
test_path  = DATA_DIR / f'bayesnf_fold{FOLD}_test.parquet'

def _ensure_local(local_path: Path, s3_key: str) -> None:
    if local_path.exists():
        print(f'  cache hit: {local_path.name} ({local_path.stat().st_size/1e6:.0f} MB)')
        return
    import boto3
    print(f'  downloading s3://{S3_BUCKET}/{s3_key} ...')
    boto3.client('s3').download_file(S3_BUCKET, s3_key, str(local_path))
    print(f'  ↓ {local_path.name} ({local_path.stat().st_size/1e6:.0f} MB)')

_ensure_local(train_path, f'{DATA_S3_DIR}/bayesnf_fold{FOLD}_train.parquet')
_ensure_local(test_path,  f'{DATA_S3_DIR}/bayesnf_fold{FOLD}_test.parquet')

date_filter = [
    ('datetime', '>=', pd.Timestamp(f'{YEAR_START}-01-01')),
    ('datetime', '<=', pd.Timestamp(f'{YEAR_END}-12-31')),
]

# Все колонки, которые могут пригодиться в любом variant:
ALL_GEO_COLS = [
    'datetime', 'latitude', 'longitude', 'x_proj', 'y_proj', 'elevation_m',
    'idw', 'gos',
    'svd_05', 'svd_06', 'svd_07', 'svd_08', 'svd_09',
    'bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa',
]
KEEP_COLS = ALL_GEO_COLS + ['rainfall', 'rainfall_int', 'station_id']

df_train = pd.read_parquet(train_path, filters=date_filter)[KEEP_COLS].copy()
df_test  = pd.read_parquet(test_path,  filters=date_filter)[KEEP_COLS].copy()
df_train['datetime'] = pd.to_datetime(df_train['datetime'])
df_test ['datetime'] = pd.to_datetime(df_test ['datetime'])

assert df_train[ALL_GEO_COLS].isna().sum().sum() == 0, 'NaN in geo cols'
assert df_test [ALL_GEO_COLS].isna().sum().sum() == 0
assert 'rainfall_int' in df_train.columns, 'ZINB needs rainfall_int = round(10·mm)'

print(f'train rows / stations : {len(df_train):>10,} / {df_train["station_id"].nunique()}')
print(f'test  rows / stations : {len(df_test):>10,} / {df_test ["station_id"].nunique()}')
print(f'dates                 : {df_train["datetime"].min().date()} → {df_train["datetime"].max().date()}')
print(f'rainfall (mm) range   : [{df_train["rainfall"].min():.2f}, {df_train["rainfall"].max():.2f}]')
print(f'rainfall_int range    : [{df_train["rainfall_int"].min()}, {df_train["rainfall_int"].max()}]')
print(f'%zero (rainfall_int)  : {100*(df_train["rainfall_int"]==0).mean():.1f}%')
print(f'\nFFRK feature stats:')
for c in ['idw', 'gos', 'svd_05', 'svd_09']:
    print(f'  {c:8s}: mean={df_train[c].mean():+.3f}  std={df_train[c].std():.3f}')
df_train.head()


## 5. JIT-cache patch для `predict`

Тот же патч что в time_seasonality / likelihood-ноутбуках. Без него `predict_bnf`
перекомпилирует closure на каждый chunk.


In [ ]:
import bayesnf.inference as bnf_inference
import bayesnf.models as bnf_models
import bayesnf.spatiotemporal as bnf_st


def _patched_predict(self, table, quantiles=(0.5,), approximate_quantiles=False):
    test_data = self.data_handler.get_test(table)
    distribution = bnf_models.LikelihoodDist(self.observation_model)

    if getattr(self, '_cached_forecast_inner', None) is None:
        model_args = self._model_args(test_data.shape)
        fn = bnf_inference._make_forecast_inner(model_args, distribution)
        for _ in range(self._ensemble_dims - 1):
            fn = jax.vmap(fn, in_axes=(0, None))
        fn = jax.pmap(fn, in_axes=(0, None))
        self._cached_forecast_inner = fn

    forecast_inner = self._cached_forecast_inner
    axis = tuple(range(self._ensemble_dims))

    forecast_params = bnf_inference.forecast_parameters_batched(
        test_data, self.params_, distribution, forecast_inner,
    )

    if distribution == bnf_models.LikelihoodDist.NORMAL:
        means, scales = forecast_params
        forecast_means = means
        forecast_quantiles = bnf_inference._get_percentile_normal(
            forecast_means, scales, quantiles, axis=axis,
            approximate=approximate_quantiles,
        )
    elif distribution in (bnf_models.LikelihoodDist.NB, bnf_models.LikelihoodDist.ZINB):
        obs_d = bnf_inference._build_observation_distribution(distribution, forecast_params)
        forecast_means = obs_d.mean()
        forecast_quantiles = jax.vmap(
            lambda q: bnf_inference._get_nb_quantiles_root(obs_d, q, ensemble_axes=axis)
        )(jnp.array(quantiles))
    else:
        raise ValueError(f'Unknown distribution: {distribution}')

    return forecast_means, forecast_quantiles


bnf_st.BayesianNeuralFieldEstimator.predict = _patched_predict
print('✓ patched BayesianNeuralFieldEstimator.predict')


## 6. Helper: `train_eval(name, feature_cols, interactions, seasonality_periods, num_seasonal_harmonics)`

Одна функция — одно полное прохождение `fit → predict → metrics`. Возвращает
словарь с метриками и сохраняет в `OUT_DIR/<name>/` (preds.parquet, metrics.json,
losses.npy). Список всех результатов лежит в глобальном `RESULTS`.

**Отличие от time_seasonality-helper:** дополнительно принимает `feature_cols` и
`interactions` — это то, что мы варьируем здесь.


In [ ]:
from properscoring import crps_ensemble

QUANTILES = (0.05, 0.1, 0.2, 0.3, 0.4, 0.50, 0.6, 0.7, 0.8, 0.9, 0.95)
QUANTILE_LABELS = [5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95]
PRED_CHUNK = 500_000
WET_THRESHOLD_MM = 0.5

# индексы нужных квантилей внутри QUANTILES (для cov / wet-metrics)
Q05_I = QUANTILE_LABELS.index(5)
Q10_I = QUANTILE_LABELS.index(10)
Q50_I = QUANTILE_LABELS.index(50)
Q90_I = QUANTILE_LABELS.index(90)
Q95_I = QUANTILE_LABELS.index(95)

RESULTS: list[dict] = []


def train_eval(
    name: str,
    feature_cols: list,
    interactions: list,
    seasonality_periods: list,
    num_seasonal_harmonics: list,
) -> dict:
    assert len(seasonality_periods) == len(num_seasonal_harmonics), \
        f'len mismatch: periods={seasonality_periods}, harmonics={num_seasonal_harmonics}'
    assert feature_cols[0] == 'datetime', 'datetime must be first feature (idx 0)'
    for i, j in interactions:
        assert 0 <= i < len(feature_cols) and 0 <= j < len(feature_cols), \
            f'interaction ({i},{j}) out of range for {len(feature_cols)} features'

    print(f'\n{"="*78}')
    print(f'  [{name}]  seasonality={seasonality_periods}/{num_seasonal_harmonics}')
    print(f'           features ({len(feature_cols)}): {feature_cols}')
    print(f'           interactions: {interactions}')
    print(f'{"="*78}')

    sub_dir = OUT_DIR / name
    sub_dir.mkdir(parents=True, exist_ok=True)

    standardize_cols = [c for c in feature_cols if c != 'datetime']

    # --- build model ---
    model = BayesianNeuralFieldVI(
        width=WIDTH, depth=DEPTH,
        freq='D', timetype='index',
        seasonality_periods=seasonality_periods,
        num_seasonal_harmonics=num_seasonal_harmonics,
        feature_cols=feature_cols,
        target_col=TARGET_COL,
        standardize=standardize_cols,
        observation_model=LIKELIHOOD,
        interactions=interactions,
    )

    # subset df_train / df_test к этому набору фич (чтобы data_handler не подавился лишним)
    keep = feature_cols + ['rainfall', 'rainfall_int', 'station_id']
    df_train_sub = df_train[keep]
    df_test_sub  = df_test [keep]

    # --- fit with heartbeat ---
    stop = threading.Event()
    def _beat():
        t0 = time.time()
        while not stop.wait(20.0):
            print(f'    [{name}] still training ({time.time()-t0:.0f}s)', flush=True)
    thr = threading.Thread(target=_beat, daemon=True); thr.start()

    t0 = time.time()
    try:
        model = model.fit(
            df_train_sub,
            seed=jax.random.PRNGKey(SEED),
            num_epochs=NUM_EPOCHS,
            ensemble_size=ENSEMBLE_SIZE,
            learning_rate=LEARNING_RATE,
            batch_size=BATCH_SIZE,
            kl_weight=KL_WEIGHT,
            sample_size_posterior=SAMPLE_SIZE_POSTERIOR,
        )
    finally:
        stop.set(); thr.join(timeout=1)
    fit_time_s = time.time() - t0
    print(f'  ✓ fit done in {fit_time_s:.0f}s ({fit_time_s/60:.1f} min)')

    losses_arr = np.asarray(model.losses_)
    np.save(sub_dir / 'losses.npy', losses_arr)

    # --- predict by chunks ---
    means_chunks, q_chunks = [], []
    t0 = time.time()
    n = len(df_test_sub)
    for i in range(0, n, PRED_CHUNK):
        j = min(i + PRED_CHUNK, n)
        m, q = model.predict(df_test_sub.iloc[i:j], quantiles=QUANTILES, approximate_quantiles=True)
        means_chunks.append(np.asarray(m))
        q_chunks.append(np.asarray(q))
    gc.collect()
    predict_time_s = time.time() - t0
    print(f'  ✓ predict {n:,} rows in {predict_time_s:.0f}s')

    means_raw = np.concatenate(means_chunks, axis=-1)
    q_raw     = np.concatenate(q_chunks, axis=-1)
    mean_per_point = means_raw.reshape(-1, n).mean(axis=0)
    q_flat = q_raw.reshape(q_raw.shape[0], -1, n).mean(axis=1)

    # --- rescale int counts → mm (ZINB target = rainfall_int) ---
    if NEEDS_RESCALE:
        mean_per_point = mean_per_point / PRECIP_SCALE
        q_flat = q_flat / PRECIP_SCALE

    # --- metrics (всё в мм; y_true берём из rainfall, не rainfall_int) ---
    y_true = df_test_sub['rainfall'].to_numpy()
    q_arr = np.stack(list(q_flat), axis=1)
    q_arr = np.sort(q_arr, axis=1)   # на случай неупорядоченных оценок квантилей

    lo90 = q_arr[:, Q05_I]; hi90 = q_arr[:, Q95_I]
    lo80 = q_arr[:, Q10_I]; hi80 = q_arr[:, Q90_I]

    metrics = {
        'name': name,
        'seasonality_periods': seasonality_periods,
        'num_seasonal_harmonics': num_seasonal_harmonics,
        'feature_cols': feature_cols,
        'interactions': interactions,
        'n_features': len(feature_cols),
        'n_quantiles': len(QUANTILES),
        'rmse'    : float(np.sqrt(np.mean((mean_per_point - y_true) ** 2))),
        'mae'     : float(np.mean(np.abs(mean_per_point - y_true))),
        'bias'    : float(np.mean(mean_per_point - y_true)),
        'cov90'   : float(np.mean((y_true >= lo90) & (y_true <= hi90))),
        'cov80'   : float(np.mean((y_true >= lo80) & (y_true <= hi80))),
        'crps'    : float(crps_ensemble(y_true, q_arr).mean()),
        'n_total' : int(len(y_true)),
        'fit_time_s'    : float(fit_time_s),
        'predict_time_s': float(predict_time_s),
    }
    wet = y_true >= WET_THRESHOLD_MM
    if wet.sum() > 0:
        metrics['mae_wet']   = float(np.mean(np.abs(mean_per_point[wet] - y_true[wet])))
        metrics['rmse_wet']  = float(np.sqrt(np.mean((mean_per_point[wet] - y_true[wet]) ** 2)))
        metrics['crps_wet']  = float(crps_ensemble(y_true[wet], q_arr[wet]).mean())
        metrics['cov90_wet'] = float(np.mean((y_true[wet] >= lo90[wet]) & (y_true[wet] <= hi90[wet])))
        metrics['n_wet']     = int(wet.sum())
    else:
        metrics.update({'mae_wet': float('nan'), 'rmse_wet': float('nan'),
                        'crps_wet': float('nan'), 'cov90_wet': float('nan'),
                        'n_wet': 0})

    # --- save ---
    with open(sub_dir / 'metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)

    preds_df = df_test_sub[['station_id', 'datetime']].copy()
    preds_df['observed_mm'] = y_true
    preds_df['mean_mm']     = mean_per_point
    for lbl, qv in zip(QUANTILE_LABELS, q_flat):
        preds_df[f'q{lbl:02d}'] = qv
    preds_df.to_parquet(sub_dir / 'preds.parquet', index=False)

    # --- print summary ---
    print(f'  CRPS     = {metrics["crps"]:.4f}    CRPS_wet = {metrics["crps_wet"]:.4f}')
    print(f'  MAE      = {metrics["mae"]:.4f}    MAE_wet  = {metrics["mae_wet"]:.4f}')
    print(f'  RMSE     = {metrics["rmse"]:.4f}   RMSE_wet = {metrics["rmse_wet"]:.4f}')
    print(f'  bias     = {metrics["bias"]:+.4f}  cov90    = {metrics["cov90"]:.3f}  cov80 = {metrics["cov80"]:.3f}')

    # cleanup GPU memory
    del model
    jax.clear_caches()
    gc.collect()

    RESULTS.append(metrics)
    return metrics


# --- Готовые feature_cols + interactions per geo-variant ---
# Каждый dict: feature_cols = ['datetime', ...spatial..., ...soil...],
# interactions ссылаются на ПОЗИЦИИ в feature_cols.
# Базовые soil-фичи всегда последние и не участвуют в interactions
# (BayesNF строит для них прямые dense-layers, без явных пар).

GEO_VARIANTS = {
    # idx: 0=time, 1=lat, 2=lon, 3=elev, 4..=soil
    'base': dict(
        feature_cols=['datetime', 'latitude', 'longitude', 'elevation_m',
                      'bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa'],
        interactions=[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)],
    ),
    # idx: 0=time, 1=lat, 2=lon, 3=x_proj, 4=y_proj, 5=elev, 6..=soil
    'proj': dict(
        feature_cols=['datetime', 'latitude', 'longitude', 'x_proj', 'y_proj',
                      'elevation_m',
                      'bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa'],
        interactions=[(0, 1), (0, 2), (0, 5), (1, 2), (1, 5), (2, 5), (3, 4)],
    ),
    # idx: 0=time, 1=lat, 2=lon, 3=elev, 4=idw, 5..=soil
    'idw': dict(
        feature_cols=['datetime', 'latitude', 'longitude', 'elevation_m', 'idw',
                      'bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa'],
        interactions=[(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (1, 3), (2, 3)],
    ),
    # idx: 0=time, 1=lat, 2=lon, 3=elev, 4..8=svd_05..09, 9..=soil
    'svd5': dict(
        feature_cols=['datetime', 'latitude', 'longitude', 'elevation_m',
                      'svd_05', 'svd_06', 'svd_07', 'svd_08', 'svd_09',
                      'bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa'],
        interactions=[(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (1, 3), (2, 3)],
    ),
    # idx: 0=time, 1=lat, 2=lon, 3=elev, 4=idw, 5=gos, 6..10=svd, 11..=soil
    'ffrk_full': dict(
        feature_cols=['datetime', 'latitude', 'longitude', 'elevation_m',
                      'idw', 'gos',
                      'svd_05', 'svd_06', 'svd_07', 'svd_08', 'svd_09',
                      'bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa'],
        interactions=[(0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (1, 2), (1, 3), (2, 3)],
    ),
}

print('✓ helper ready, GEO_VARIANTS:', list(GEO_VARIANTS.keys()))


## 7. Train — `WY_h1_10 × ffrk_full`

Лучшая сезонность (`WY_h1_10`) плюс полный FFRK-блок (idw + gos + svd_05..09).
Это основная модель, под которую написаны все аналитические ячейки ниже.


In [ ]:
v = GEO_VARIANTS['ffrk_full']
m = train_eval(
    name='WY_h1_10__ffrk_full',
    feature_cols=v['feature_cols'], interactions=v['interactions'],
    seasonality_periods=SEAS_WY_H1_10['periods'],
    num_seasonal_harmonics=SEAS_WY_H1_10['harmonics'],
)


## 8. Train — `Y_h10 × ffrk_full` (control)

Контроль: тот же `ffrk_full`, но без weekly harmonic. Сравнение даёт оценку
того, насколько FFRK-фичи «заменяют» weekly-компоненту.


In [ ]:
v = GEO_VARIANTS['ffrk_full']
m = train_eval(
    name='Y_h10__ffrk_full',
    feature_cols=v['feature_cols'], interactions=v['interactions'],
    seasonality_periods=SEAS_Y_H10['periods'],
    num_seasonal_harmonics=SEAS_Y_H10['harmonics'],
)


## 9. Загрузка артефактов

Все ячейки ниже самодостаточны: подгружают то, что записал `train_eval` в
`results/bayesnf/experiments/geo_features/WY_h1_10__ffrk_full/`:

- `metrics.json` — глобальные метрики;
- `preds.parquet` — `(station_id, datetime, observed_mm, mean_mm, q05, q50, q95)`;
- `losses.npy` — ELBO по эпохам для каждого члена ансамбля.

Плюс заново читаем `df_test` (lat/lon по `station_id` для пространственных диагностик).


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EXP_NAME = 'WY_h1_10__ffrk_full'
EXP_DIR  = Path('results/bayesnf/experiments/geo_features') / EXP_NAME
assert EXP_DIR.exists(), f'не найдена папка эксперимента: {EXP_DIR}'

metrics = json.loads((EXP_DIR / 'metrics.json').read_text())
losses  = np.load (EXP_DIR / 'losses.npy')
# bayesnf на multi-GPU pmap возвращает losses как (n_devices, ens_per_device, n_steps);
# на 1 GPU — (ensemble_size, n_steps). Универсально схлопываем в (n_members, n_steps).
if losses.ndim > 2:
    losses = losses.reshape(-1, losses.shape[-1])
preds   = pd.read_parquet(EXP_DIR / 'preds.parquet')
preds['datetime'] = pd.to_datetime(preds['datetime'])

# координаты станций — нужны для пространственных диагностик
if 'df_test' not in globals():
    test_path = Path('results/bayesnf/data') / f'bayesnf_fold{FOLD}_test.parquet'
    df_test = pd.read_parquet(
        test_path,
        columns=['station_id','datetime','latitude','longitude','rainfall'],
        filters=[('datetime','>=', pd.Timestamp(f'{YEAR_START}-01-01')),
                 ('datetime','<=', pd.Timestamp(f'{YEAR_END}-12-31'))])
    df_test['datetime'] = pd.to_datetime(df_test['datetime'])

coords = (df_test[['station_id','latitude','longitude']]
            .drop_duplicates('station_id').set_index('station_id'))
preds = preds.merge(coords, left_on='station_id', right_index=True, how='left')

print(f'experiment        : {EXP_NAME}')
print(f'losses shape      : {losses.shape}  (ensemble × epochs)')
print(f'preds rows        : {len(preds):,}')
print(f'stations in test  : {preds["station_id"].nunique()}')
print(f'dates in test     : {preds["datetime"].nunique()}')
print(f'date range        : {preds["datetime"].min().date()} → {preds["datetime"].max().date()}')


## 10. Кривые потерь (ELBO) по ансамблю

Каждая голубая линия — один член ансамбля. Согласованность ⇒ робастность оптимизации; плато ⇒ запас по эпохам. Слева линейная шкала, справа symlog.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
mean_loss = losses.mean(0)
epochs = np.arange(losses.shape[1])
for ax, log in zip(axes, [False, True]):
    for m in range(losses.shape[0]):
        ax.plot(epochs, losses[m], color='tab:blue', alpha=0.3, lw=0.8)
    ax.plot(epochs, mean_loss, color='black', lw=1.6, label='ensemble mean')
    ax.fill_between(epochs, losses.min(0), losses.max(0),
                    color='tab:blue', alpha=0.15, label='min–max envelope')
    ax.set_xlabel('epoch'); ax.set_ylabel('ELBO loss')
    ax.grid(alpha=0.3); ax.legend()
    if log:
        ax.set_yscale('symlog'); ax.set_title('log scale')
    else:
        ax.set_title(f'{EXP_NAME} — {losses.shape[0]} members × {losses.shape[1]} epochs')
plt.tight_layout(); plt.show()

final = mean_loss[-1]; tol = 0.01 * abs(final)
converged_at = next((e for e, v in enumerate(mean_loss) if abs(v - final) <= tol),
                    losses.shape[1] - 1)
print(f'≈ сошёлся на эпохе {converged_at}/{losses.shape[1]-1} '
      f'(|loss − loss_final| ≤ 1%)')


## 11. Метрики предсказания

Глобальные / wet-only / runtime. Для контекста:
- Saad et al. 2024 на Colorado-Precipitation: RMSE 1.80, MAE 1.23, MIS 8.33;
- OK на квоте у вас: CRPS ≈ 1.0–1.5 мм;
- LGBM: CRPS ≈ 0.9–1.1 мм.


In [ ]:
def _fmt(k, v):
    if isinstance(v, float): return f'  {k:14s} = {v:>10.4f}'
    if isinstance(v, int):   return f'  {k:14s} = {v:>10,d}'
    return                          f'  {k:14s} = {v}'

ORDER_GLOBAL = ['rmse','mae','bias','crps','cov90','cov80','n_total','n_quantiles']
ORDER_WET    = ['rmse_wet','mae_wet','crps_wet','cov90_wet','n_wet']
ORDER_TIME   = ['fit_time_s','predict_time_s']

print('═'*42, '  GLOBAL  ', '═'*42)
for k in ORDER_GLOBAL: print(_fmt(k, metrics[k]))
print('\n' + '═'*42 + '  WET-ONLY' + '═'*42)
for k in ORDER_WET:    print(_fmt(k, metrics[k]))
print('\n' + '═'*42 + '  RUNTIME ' + '═'*42)
for k in ORDER_TIME:
    v = metrics[k]
    print(f'  {k:14s} = {v:>10.1f} s   ({v/60:.1f} min)')

print(f'\n  spec : seasonality={metrics["seasonality_periods"]} / '
      f'h={metrics["num_seasonal_harmonics"]}')
print(f'         features ({metrics["n_features"]}): {metrics["feature_cols"]}')


## 12. Калибровка интервальных предсказаний

Saad et al. 2024 главным образом гордится **calibrated 95% PIs**. У нас 11 квантилей
(`q05, q10, …, q95`), поэтому делаем три полноценных диагностики:

1. **Reliability curve** — нормативное α vs эмпирическое `P(y ≤ q_α)` по всем 11 уровням.
2. **Coverage 90% по магнитуде** — попадание в `[q05, q95]` по децилям `y`. Узкие интервалы на heavy tails — типичный артефакт ZINB.
3. **Spread–skill** — ширина интервала `q95 − q05` vs `|q50 − y|`. У хорошей модели — корреляция.


In [ ]:
y   = preds['observed_mm'].to_numpy()
q50 = preds['q50'].to_numpy()
q05 = preds['q05'].to_numpy()
q95 = preds['q95'].to_numpy()

# Собираем все квантили из preds (q05, q10, ..., q95)
Q_LABELS = [5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95]
Q_LEVELS = np.array([l / 100 for l in Q_LABELS])
q_cols = [f'q{l:02d}' for l in Q_LABELS]
Q = np.stack([preds[c].to_numpy() for c in q_cols], axis=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (1) reliability — 11-point curve
emp = np.array([(y <= Q[:, k]).mean() for k in range(len(Q_LABELS))])
ax = axes[0]
ax.plot([0,1],[0,1],'--', color='grey', alpha=0.5, label='perfect')
ax.plot(Q_LEVELS, emp, 'o-', ms=6, color='tab:red', label='empirical')
for lvl, e in zip(Q_LEVELS, emp):
    if lvl in (0.05, 0.5, 0.95):
        ax.annotate(f'{e:.3f}', (lvl, e), textcoords='offset points', xytext=(6,-4),
                    fontsize=8)
ax.set_xlabel('nominal α'); ax.set_ylabel('empirical P(y ≤ qα)')
ax.set_title(f'Reliability — {len(Q_LABELS)} quantile levels')
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.grid(alpha=0.3); ax.legend()

# (2) coverage by magnitude
ax = axes[1]
in_iv = (y >= q05) & (y <= q95)
bins = np.quantile(y, np.linspace(0, 1, 11))
bin_id = np.clip(np.digitize(y, bins[1:-1]), 0, 9)
cov_by_bin = np.array([in_iv[bin_id == k].mean() if (bin_id == k).any() else np.nan
                       for k in range(10)])
ax.bar(range(10), cov_by_bin, color='tab:blue', alpha=0.8)
ax.axhline(0.9, ls='--', color='red', alpha=0.6, label='nominal 0.90')
ax.set_xticks(range(10))
ax.set_xticklabels([f'{bins[i]:.1f}–{bins[i+1]:.1f}' for i in range(10)],
                   rotation=45, ha='right')
ax.set_xlabel('y_true decile (mm)'); ax.set_ylabel('coverage of [q05, q95]')
ax.set_title(f'Coverage 90% by magnitude (overall = {in_iv.mean():.3f})')
ax.legend(); ax.grid(alpha=0.3, axis='y')

# (3) spread–skill
ax = axes[2]
width = q95 - q05
err   = np.abs(q50 - y)
K = 20
edges = np.quantile(width, np.linspace(0, 1, K+1))
centers = 0.5 * (edges[:-1] + edges[1:])
ids = np.clip(np.digitize(width, edges[1:-1]), 0, K-1)
mean_err = np.array([err[ids == k].mean() if (ids == k).any() else np.nan
                     for k in range(K)])
ax.plot(centers, mean_err, 'o-', color='tab:purple')
lim = max(np.nanmax(centers), np.nanmax(mean_err))
ax.plot([0, lim], [0, lim], '--', color='grey', alpha=0.5, label='y=x')
ax.set_xlabel('interval width  q95 − q05  (mm)')
ax.set_ylabel('|q50 − y|  (mm)')
ax.set_title('Spread–skill')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

print('Reliability:')
for lvl, e in zip(Q_LEVELS, emp):
    bias = e - lvl
    marker = '  ✓' if abs(bias) < 0.02 else ('  ↑' if bias > 0 else '  ↓')
    print(f'  α={lvl:.2f}  emp={e:.3f}  Δ={bias:+.3f}{marker}')
print(f'\nCoverage 90% (q05..q95) : overall {in_iv.mean():.3f}  '
      f'(low-decile {cov_by_bin[0]:.3f},  high-decile {cov_by_bin[-1]:.3f})')


## 13. Observed vs Predicted (scatter, остатки, карта по станциям)

Три панели:
1. **Scatter** mean vs observed (`hexbin`, log-плотность);
2. **Гистограмма остатков** `mean − observed`;
3. **Карта по станциям**: цвет = MAE на станции (как Saad Fig. 4d).


In [ ]:
y    = preds['observed_mm'].to_numpy()
yhat = preds['mean_mm'    ].to_numpy()
resid = yhat - y

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.hexbin(y, yhat, gridsize=60, mincnt=1, bins='log', cmap='viridis')
lim = max(y.max(), yhat.max())
ax.plot([0, lim], [0, lim], '--', color='red', alpha=0.6, lw=1.5)
ax.set_xlabel('observed (mm)'); ax.set_ylabel('predicted mean (mm)')
ax.set_title('Observed vs Predicted (log-density)')
ax.set_xlim(0, lim); ax.set_ylim(0, lim); ax.grid(alpha=0.3)

ax = axes[1]
qs = np.quantile(resid, [0.005, 0.995])
ax.hist(resid, bins=80, range=qs, color='tab:grey', alpha=0.85)
ax.axvline(0, color='red', ls='--', alpha=0.6)
ax.set_xlabel('predicted − observed (mm)'); ax.set_ylabel('count')
ax.set_title(f'Residuals  μ={resid.mean():+.3f}  σ={resid.std():.3f}')
ax.grid(alpha=0.3, axis='y')

ax = axes[2]
per_st = (preds.assign(abs_err=np.abs(yhat - y))
                .groupby('station_id')
                .agg(lat=('latitude','first'), lon=('longitude','first'),
                     mae=('abs_err','mean'),    n=('abs_err','size')))
sc = ax.scatter(per_st['lon'], per_st['lat'], c=per_st['mae'],
                s=18, cmap='magma_r', edgecolor='black', linewidth=0.3)
plt.colorbar(sc, ax=ax, label='per-station MAE (mm)')
ax.set_xlabel('longitude (°)'); ax.set_ylabel('latitude (°)')
ax.set_title(f'Spatial MAE — {len(per_st)} test stations')
ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

print('top-5 worst stations:')
print(per_st.sort_values('mae', ascending=False).head(5).to_string())


## 14. Time-series по 4 примерным станциям (Saad Fig. 3 style)

Чёрные точки — observed; красная — `mean_mm`; серая лента — `[q05, q95]`. Берём worst-MAE, median, best и одну случайную станцию.


In [ ]:
per_st = (preds.assign(abs_err=np.abs(preds['mean_mm'] - preds['observed_mm']))
                .groupby('station_id').agg(mae=('abs_err','mean')))
rng = np.random.default_rng(0)
picks = {
    'worst MAE' : per_st['mae'].idxmax(),
    'median MAE': per_st['mae'].sort_values().index[len(per_st)//2],
    'best MAE'  : per_st['mae'].idxmin(),
    'random'    : rng.choice(per_st.index),
}

fig, axes = plt.subplots(len(picks), 1,
                         figsize=(13, 2.4*len(picks)), sharex=True)
for ax, (label, sid) in zip(axes, picks.items()):
    s = preds[preds.station_id == sid].sort_values('datetime')
    ax.fill_between(s['datetime'], s['q05'], s['q95'],
                    color='lightgrey', label='95% PI')
    ax.plot(s['datetime'], s['mean_mm'], color='tab:red', lw=1.0,
            label='BayesNF mean')
    ax.scatter(s['datetime'], s['observed_mm'], color='black', s=6,
               label='observed', alpha=0.7)
    ax.set_ylabel('mm')
    ax.set_title(f'{label}: station={sid} (MAE={per_st.loc[sid,"mae"]:.2f} mm)')
    ax.grid(alpha=0.3)
axes[0].legend(loc='upper right', ncol=3)
plt.tight_layout(); plt.show()


## 15. Семивариограммный диагностик (Saad et al. 2024, Fig. 5)

Sanity-check на пространственно-временную структуру ковариации. По Saad eq. 14:
$2\gamma(h, \tau) = \mathrm{Var}\bigl[Y(s+h,\,t+\tau) - Y(s,\,t)\bigr]$.

- **Empirical** γ̂ считается на `observed_mm` тестовых станций;
- **Inferred** γ̂ — тем же методом-моментов на `mean_mm` (BayesNF prediction).

В оригинальной Saad-Fig.5 «inferred» оценивается в 70 случайных novel-точках, для чего нужен обученный объект модели; так как `train_eval` его удаляет — мы используем предсказания в тех же тестовых точках. Это адаптация диагностика: согласие двух поверхностей по-прежнему означает, что модель воспроизводит структуру γ; рассогласование на малых τ типично интерпретируется как high-frequency day-to-day noise, поглощённый θ_y.

Bins: `h ∈ [0, 500]` km шаг 50 km; `τ ∈ {0,…,10}` дней.


In [ ]:
DIST_EDGES_KM = np.arange(0, 525, 50)
TIME_LAGS     = np.arange(0, 11)

stations = sorted(preds['station_id'].unique())
S = len(stations)

lat = np.array([coords.loc[s, 'latitude']  for s in stations])
lon = np.array([coords.loc[s, 'longitude'] for s in stations])
R = 6371.0
lat_r = np.radians(lat); lon_r = np.radians(lon)
dlat = lat_r[:, None] - lat_r[None, :]
dlon = lon_r[:, None] - lon_r[None, :]
a = np.sin(dlat/2)**2 + np.cos(lat_r)[:, None]*np.cos(lat_r)[None, :]*np.sin(dlon/2)**2
D_km = 2*R*np.arcsin(np.sqrt(np.clip(a, 0, 1)))

iu, ju = np.triu_indices(S, k=1)
d_pair = D_km[iu, ju]
bin_id = np.clip(np.digitize(d_pair, DIST_EDGES_KM[1:-1]), 0, len(DIST_EDGES_KM)-2)
nb = len(DIST_EDGES_KM) - 1

obs_panel  = (preds.pivot_table('observed_mm', index='datetime', columns='station_id')
                   .reindex(columns=stations).sort_index())
pred_panel = (preds.pivot_table('mean_mm',     index='datetime', columns='station_id')
                   .reindex(columns=stations).sort_index())
OBS  = obs_panel .to_numpy(dtype=np.float32)
PRED = pred_panel.to_numpy(dtype=np.float32)

def _semivariogram(panel):
    out = np.full((len(TIME_LAGS), nb), np.nan, dtype=np.float64)
    bin_flat = np.broadcast_to(bin_id[None, :], (1, len(iu)))[0]
    for k, tau in enumerate(TIME_LAGS):
        A = panel       if tau == 0 else panel[:-tau]
        B = panel       if tau == 0 else panel[tau:]
        diff = A[:, iu] - B[:, ju]                  # (T-τ, P)
        sq = (diff * diff).astype(np.float64)
        mask = np.isfinite(sq)
        if not mask.any():
            continue
        flat_sq = sq[mask]
        flat_bin = np.broadcast_to(bin_flat[None, :], sq.shape)[mask]
        sums = np.bincount(flat_bin, weights=flat_sq, minlength=nb)
        cnts = np.bincount(flat_bin, minlength=nb).astype(np.float64)
        with np.errstate(invalid='ignore'):
            out[k] = np.where(cnts > 0, 0.5 * sums / np.maximum(cnts, 1), np.nan)
    return out

print('computing empirical semivariogram ...', flush=True)
gamma_obs = _semivariogram(OBS)
print('computing inferred  semivariogram ...', flush=True)
gamma_inf = _semivariogram(PRED)
print(f'shapes: obs {gamma_obs.shape}, inf {gamma_inf.shape}')

# ── 3D surfaces (Saad Fig. 5a) ─────────────────────────────────────
H = 0.5*(DIST_EDGES_KM[:-1] + DIST_EDGES_KM[1:])
HH, TT = np.meshgrid(H, TIME_LAGS)
fig = plt.figure(figsize=(14, 5))
for sub, (G, title) in enumerate(
        [(gamma_obs, 'Empirical semivariogram (observed)'),
         (gamma_inf, 'Inferred semivariogram (BayesNF mean)')], 1):
    ax = fig.add_subplot(1, 2, sub, projection='3d')
    surf = ax.plot_surface(HH, TT, G, cmap='viridis',
                           edgecolor='k', linewidth=0.2, alpha=0.95)
    ax.set_xlabel('distance (km)')
    ax.set_ylabel('time lag (days)')
    ax.set_zlabel('γ')
    ax.set_title(title)
    fig.colorbar(surf, ax=ax, shrink=0.6, pad=0.1)
plt.tight_layout(); plt.show()

# ── sliced panels (Saad Fig. 5b) ───────────────────────────────────
fig, axes = plt.subplots(2, 6, figsize=(16, 5.5), sharex=True, sharey=True)
for k, tau in enumerate(TIME_LAGS):
    ax = axes.flat[k]
    ax.plot(H, gamma_obs[k], '-',  color='black',   lw=1.5, label='Empirical')
    ax.plot(H, gamma_inf[k], '--', color='tab:red', lw=1.5, label='Inferred')
    ax.set_title(f'τ = {tau} days')
    ax.grid(alpha=0.3)
    if k == 0: ax.legend()
    if k % 6 == 0: ax.set_ylabel('semivariance γ')
    if k // 6 == 1: ax.set_xlabel('distance (km)')
axes.flat[-1].axis('off')
plt.tight_layout(); plt.show()

with np.errstate(divide='ignore', invalid='ignore'):
    rel = np.abs(gamma_inf - gamma_obs) / np.where(gamma_obs > 0, gamma_obs, np.nan)
print(f'mean |γ_inf − γ_obs| / γ_obs over (h,τ)        = {np.nanmean(rel):.3f}')
print(f'  short lags τ ∈ {{0,1,2}}                       = {np.nanmean(rel[:3]):.3f}')
print(f'  long  lags τ ∈ {{3..10}}                       = {np.nanmean(rel[3:]):.3f}')
